# 02 — Обучение одной CNN на RuASD (GPU)
Блокнот использует FLAC и `ruasd_segments_manifest.csv`, созданные notebook 01. Log-Mel вычисляются на лету и не сохраняются как изображения. `test_unseen` не участвует в обучении, ранней остановке или выборе checkpoint.

In [ ]:
# 1. Среда, репозиторий и зависимости
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Unvand/audio-deepfake-detector.git'
REPO_DIR = Path('/content/audio-deepfake-detector')
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab-train.txt')], check=True)
    os.chdir(REPO_DIR)
print('Python:', sys.version)

In [ ]:
# 2. Drive, GPU и подготовленный manifest
import pandas as pd
import torch

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
PREPARED_ROOT = Path('/content/drive/MyDrive/VKR_RuASD/prepared_v1')
TRAINING_ROOT = Path('/content/drive/MyDrive/VKR_RuASD/training_v1')
SEGMENTS_MANIFEST = PREPARED_ROOT / 'ruasd_segments_manifest.csv'
if not SEGMENTS_MANIFEST.exists():
    raise FileNotFoundError('Сначала полностью выполните notebook 01: ' + str(SEGMENTS_MANIFEST))
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('Выберите Runtime → Change runtime type → GPU')
segments = pd.read_csv(SEGMENTS_MANIFEST)
display(pd.crosstab(segments['split'], segments['label'], margins=True))
assert not segments['segment_id'].duplicated().any()

In [ ]:
# 3. Фиксируем log-Mel и параметры обучения
from audio_deepfake_detector.src.cnn_training import (
    MelConfig,
    RuASDSegmentDataset,
    TrainingConfig,
    evaluate_checkpoint,
    train_cnn,
)

MEL_CONFIG = MelConfig(
    sample_rate=32_000, n_fft=1_024, win_length=1_024,
    hop_length=320, n_mels=80, f_min=20.0, f_max=16_000.0,
)
TRAIN_CONFIG = TrainingConfig(
    batch_size=32, epochs=20, learning_rate=1e-3,
    weight_decay=1e-4, num_workers=2, seed=42,
    early_stopping_patience=5,
)
print(MEL_CONFIG)
print(TRAIN_CONFIG)

In [ ]:
# 4. Необязательная smoke-проверка одного train-примера (без обучения)
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    smoke_dataset = RuASDSegmentDataset(segments, PREPARED_ROOT, 'train', MEL_CONFIG)
    log_mel, label, sample_id = smoke_dataset[0]
    print('log-Mel:', tuple(log_mel.shape), 'label:', label.item(), 'sample_id:', sample_id)
    assert torch.isfinite(log_mel).all()

## Обучение и возобновление
После каждой эпохи атомарно обновляются `last_checkpoint.pth` и история. Лучший по source-level validation F1 checkpoint сохраняется как `best_model.pth`. Повторный запуск этой ячейки с `RESUME=True` продолжит со следующей эпохи.

In [ ]:
# 5. Явный запуск обучения; по умолчанию выключен
RUN_TRAINING = False
RESUME = True
if RUN_TRAINING:
    model, history = train_cnn(
        PREPARED_ROOT, TRAINING_ROOT, MEL_CONFIG, TRAIN_CONFIG, resume=RESUME
    )
    print('Завершённых эпох:', len(history))
    print('Best:', TRAINING_ROOT / 'best_model.pth')
    print('Last:', TRAINING_ROOT / 'last_checkpoint.pth')
else:
    print('Обучение не запущено. Установите RUN_TRAINING = True.')

In [ ]:
# 6. История Accuracy/Precision/Recall/F1/EER на validation
import json

import matplotlib.pyplot as plt

history_path = TRAINING_ROOT / 'training_history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    display(pd.json_normalize(history))
    history_frame = pd.json_normalize(history)
    history_frame.plot(x='epoch', y=['validation_source.f1', 'validation_source.eer'], marker='o')
    plt.grid(True)
    plt.show()
else:
    print('История появится после первой завершённой эпохи.')

In [ ]:
# 7. Финальная оценка test_unseen — только после выбора модели
FINAL_EVALUATE_TEST_UNSEEN = False
if FINAL_EVALUATE_TEST_UNSEEN:
    best_model = TRAINING_ROOT / 'best_model.pth'
    if not best_model.exists():
        raise FileNotFoundError(best_model)
    final_metrics = evaluate_checkpoint(
        PREPARED_ROOT, best_model, split='test_unseen',
        batch_size=TRAIN_CONFIG.batch_size, num_workers=TRAIN_CONFIG.num_workers,
    )
    print(final_metrics)
    (TRAINING_ROOT / 'test_unseen_metrics.json').write_text(
        json.dumps(final_metrics, ensure_ascii=False, indent=2), encoding='utf-8'
    )
else:
    print('test_unseen не использован.')